# Tarefa 2 - Versao Aprendida em Aula

Esta versao tenta ficar o mais proxima possivel do estilo mostrado nas aulas da professora:
- `webdriver.Chrome()`
- `WebDriverWait`
- `presence_of_all_elements_located`
- seletores simples com `By.CSS_SELECTOR`, `By.TAG_NAME` e `By.XPATH`
- abrir pagina em nova aba
- salvar os resultados em JSON


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import re
import time

# Para teste rapido, voce pode trocar 250 por 5.
TOTAL_FILMES = 250

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

driver.get("https://www.imdb.com/pt/chart/top/")

filmes = wait.until(
    EC.presence_of_all_elements_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

print("Quantidade de cards encontrados:", len(filmes))

aba_principal = driver.current_window_handle
dados = []


Quantidade de cards encontrados: 125


In [2]:
for posicao, filme in enumerate(filmes[:TOTAL_FILMES], start=1):
    try:
        url = filme.find_element(By.TAG_NAME, "a").get_attribute("href")
    except:
        url = None

    if url is None:
        continue

    print(f"Coletando {posicao}/{TOTAL_FILMES}: {url}")

    titulo = None
    ano = None
    nota = None
    url_poster = None
    imagem_poster = None
    lista_generos = []
    lista_diretores = []

    driver.execute_script("window.open(arguments[0]);", url)
    driver.switch_to.window(driver.window_handles[-1])

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

        try:
            titulo = driver.find_element(By.TAG_NAME, "h1").text.strip()
        except:
            titulo = None

        try:
            meta_items = driver.find_elements(By.CSS_SELECTOR, "li.ipc-inline-list__item")
            for item in meta_items:
                texto = item.text.strip()
                if re.fullmatch(r"\d{4}", texto):
                    ano = texto
                    break
        except:
            ano = None

        try:
            nota = driver.find_element(By.CSS_SELECTOR, "span.ipc-rating-star").text.strip()
        except:
            nota = None

        try:
            poster = driver.find_element(By.TAG_NAME, "img")
            url_poster = poster.get_attribute("src")
            imagem_poster = poster.get_attribute("src")
        except:
            url_poster = None
            imagem_poster = None

        try:
            generos = wait.until(
                EC.presence_of_all_elements_located((
                    By.CSS_SELECTOR,
                    ".ipc-chip-list__scroller"
                ))
            )

            for genero in generos:
                tags_genero = genero.find_elements(By.CSS_SELECTOR, "span.ipc-chip__text")
                for tag in tags_genero:
                    texto = tag.text.strip()
                    if texto != "" and texto not in lista_generos:
                        lista_generos.append(texto)
        except:
            lista_generos = []

        try:
            creditos = wait.until(
                EC.presence_of_all_elements_located((
                    By.CSS_SELECTOR,
                    'li[data-testid="title-pc-principal-credit"]'
                ))
            )

            if len(creditos) > 0:
                bloco_direcao = creditos[0]
                tags_diretor = bloco_direcao.find_elements(By.CSS_SELECTOR, "li.ipc-inline-list__item")

                for tag in tags_diretor:
                    texto = tag.text.strip()
                    if texto != "" and texto not in lista_diretores:
                        lista_diretores.append(texto)
        except:
            lista_diretores = []

    except:
        print("Erro ao abrir a pagina do filme.")

    finally:
        driver.close()
        driver.switch_to.window(aba_principal)

    dados.append({
        "Posicao": posicao,
        "Titulo": titulo,
        "Ano": ano,
        "URL_Poster": url_poster,
        "Imagem_Poster": imagem_poster,
        "Nota_IMDb": nota,
        "Generos": lista_generos,
        "Diretores": lista_diretores,
        "URL": url
    })

    time.sleep(1)

driver.quit()

with open("filmes_imdb_aula.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, indent=2, ensure_ascii=False)

print(json.dumps(dados[:3], indent=2, ensure_ascii=False))
print("Arquivo JSON salvo com sucesso: filmes_imdb_aula.json")


Coletando 1/2: https://www.imdb.com/pt/title/tt0111161/?ref_=chttp_i_1
Coletando 2/2: https://www.imdb.com/pt/title/tt0068646/?ref_=chttp_i_2
[
  {
    "Posicao": 1,
    "Titulo": "Um Sonho de Liberdade",
    "Ano": "1994",
    "URL_Poster": "https://m.media-amazon.com/images/M/MV5BMDAyY2FhYjctNDc5OS00MDNlLThiMGUtY2UxYWVkNGY2ZjljXkEyXkFqcGc@._V1_QL75_UX190_CR0,2,190,281_.jpg",
    "Imagem_Poster": "https://m.media-amazon.com/images/M/MV5BMDAyY2FhYjctNDc5OS00MDNlLThiMGUtY2UxYWVkNGY2ZjljXkEyXkFqcGc@._V1_QL75_UX190_CR0,2,190,281_.jpg",
    "Nota_IMDb": "9,3",
    "Generos": [
      "Drama de época",
      "Drama prisional",
      "Drama"
    ],
    "Diretores": [
      "Frank Darabont"
    ],
    "URL": "https://www.imdb.com/pt/title/tt0111161/?ref_=chttp_i_1"
  },
  {
    "Posicao": 2,
    "Titulo": "O Poderoso Chefão",
    "Ano": "1972",
    "URL_Poster": "https://m.media-amazon.com/images/M/MV5BYTRmMjkwYzYtYTRiMi00NDJjLTk1NjctMDA3MjY2ZWIyMGQ5XkEyXkFqcGc@._V1_QL75_UY281_CR4,0,190,281_.j